<a href="https://colab.research.google.com/github/helenamrch/OAS5900-Masteroppgave/blob/main/ssb_navn_kjonn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SSB Navneanalyse – Kjønnsbestemmelse

Bruker SSBs PxWebApi (tabell **10467**) for å avgjøre om navn er kvinnelige, mannlige eller kjønnsnøytrale.

**Slik er tabellen strukturert:**  
Tabellen har én variabel `Fornavn` med verdikoder på formen `1EMMA`, `2OLE` osv.  
Prefikset angir kjønn: **`1` = jente/kvinne**, **`2` = gutt/mann**.

**Strategi:** To bulk-kall — ett for alle jentenavn (prefix `1`), ett for alle guttenavn (prefix `2`) —  
bygger en lokal oppslagstabell. Ingen per-navn API-kall.

In [ ]:
import os

# List contents of the root of MyDrive
drive_path = 'PLACEHOLDER'

if os.path.exists(drive_path):
    print(f"Innhold i '{drive_path}':")
    for item in os.listdir(drive_path):
        print(item)
else:
    print(f"Klarte ikke å finne '{drive_path}'. Er Google Drive montert riktig?")

In [ ]:
import os

masteroppgave_path = ''PLACEHOLDER''

if os.path.exists(masteroppgave_path):
    print(f"Innhold i '{masteroppgave_path}':")
    for item in os.listdir(masteroppgave_path):
        print(item)
else:
    print(f"Klarte ikke å finne '{masteroppgave_path}'. Sjekk banen.")

## 1. Importer biblioteker

In [ ]:
import requests
import pandas as pd
import json
from collections import defaultdict

## 2. Konfigurasjon

In [ ]:
INPUT_CSV         = "'PLACEHOLDER'"   # Excel-fil med kolonne 'navn'
OUTPUT_CSV        = "'PLACEHOLDER'"  # Resultat med tilleggskolonne 'kjønn'
TABLE_ID          = "10467"
API_URL           = f"https://data.ssb.no/api/v0/no/table/{TABLE_ID}"

# Andel minoritetskjønn som skal til for å klassifisere som kjønnsnøytralt (0.0–0.5)
# 0.25 = minst 25 % av forekomstene er av det andre kjønnet
NEUTRAL_THRESHOLD = 0.25

# Kjønnsprefikser i SSBs verdikoder
PREFIX_JENTE = "1"   # f.eks. '1EMMA'
PREFIX_GUTT  = "2"   # f.eks. '2OLE'

# ContentsCode for absolutte tall (ikke prosent)
CONTENTS_CODE = "Personer"

print(f"API-endepunkt: {API_URL}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Hent metadata og del opp navnelisten

In [ ]:
resp = requests.get(API_URL, timeout=15)
resp.raise_for_status()
meta = resp.json()

variables = {v["code"]: v for v in meta["variables"]}

# Alle Fornavn-koder og tilhørende visningsnavn
fornavn_koder = variables["Fornavn"]["values"]        # ['1AAGOT', '1AASE', ..., '2AAMUND', ...]
fornavn_tekst = variables["Fornavn"]["valueTexts"]    # ['Aagot', 'Aase', ..., 'Aamund', ...]
alle_år       = variables["Tid"]["values"]

# Del opp etter kjønnsprefiks
jente_koder = [k for k in fornavn_koder if k.startswith(PREFIX_JENTE)]
gutt_koder  = [k for k in fornavn_koder if k.startswith(PREFIX_GUTT)]

# Bygg mapping: renavn (store bokstaver) → visningsnavn
kode_til_navn = dict(zip(fornavn_koder, fornavn_tekst))

print(f"Totalt {len(fornavn_koder)} fornavn i tabellen")
print(f"  Jentenavn (prefix '{PREFIX_JENTE}'): {len(jente_koder)}")
print(f"  Guttenavn (prefix '{PREFIX_GUTT}') : {len(gutt_koder)}")
print(f"  År       : {len(alle_år)} ({alle_år[0]}–{alle_år[-1]})")
print(f"\nEksempel jentenavn-koder: {jente_koder[:5]}")
print(f"Eksempel guttenavn-koder: {gutt_koder[:5]}")

## 4. Last ned alle navn fra SSB

To bulk-kall — ett for jentenavn og ett for guttenavn.  
Resultatet er to ordbøker: `{NAVN: totalt_antall_fødte}` (summert over alle år).

In [ ]:
def fetch_name_counts(name_codes: list, alle_år: list) -> dict:
    """
    Henter totalt antall fødte (summert over alle år) for en liste navnkoder.
    Returnerer dict {NAVNKODE: total}, f.eks. {'1EMMA': 45231}.
    """
    payload = {
        "query": [
            {
                "code": "Fornavn",
                "selection": {"filter": "item", "values": name_codes}
            },
            {
                "code": "ContentsCode",
                "selection": {"filter": "item", "values": [CONTENTS_CODE]}
            },
            {
                "code": "Tid",
                "selection": {"filter": "item", "values": alle_år}
            }
        ],
        "response": {"format": "json-stat2"}
    }

    r = requests.post(API_URL, json=payload, timeout=120)
    if r.status_code != 200:
        print(f"  HTTP {r.status_code}: {r.text[:400]}")
        r.raise_for_status()

    data    = r.json()
    dims    = data["id"]
    sizes   = data["size"]
    vals    = data["value"]

    n_names = sizes[dims.index("Fornavn")]
    n_years = sizes[dims.index("Tid")]

    returned_codes = list(data["dimension"]["Fornavn"]["category"]["index"].keys())

    totals = {}
    for i, code in enumerate(returned_codes):
        start = i * n_years
        total = sum(v for v in vals[start : start + n_years] if v is not None)
        if total > 0:
            totals[code] = total

    return totals


print("Henter alle jentenavn... (kan ta 15–60 sek)")
jente_tall = fetch_name_counts(jente_koder, alle_år)
print(f"  ✓ {len(jente_tall)} jentenavn med data")

print("Henter alle guttenavn...")
gutt_tall = fetch_name_counts(gutt_koder, alle_år)
print(f"  ✓ {len(gutt_tall)} guttenavn med data")

# Topp 5 for verifisering
topp_j = sorted(jente_tall.items(), key=lambda x: -x[1])[:5]
topp_g = sorted(gutt_tall.items(),  key=lambda x: -x[1])[:5]
print("\nTopp 5 jentenavn:", [(kode_til_navn.get(k, k), v) for k, v in topp_j])
print("Topp 5 guttenavn:", [(kode_til_navn.get(k, k), v) for k, v in topp_g])

## 5. Klassifiser navn

In [ ]:
def classify_gender(name: str) -> str:
    """
    Slår opp navn i SSB-dataen.
    SSB-koder: '1' + NAVN_STORE_BOKSTAVER for jenter, '2' + ... for gutter.
    """
    key = name.strip().upper()
    jente_key = PREFIX_JENTE + key
    gutt_key  = PREFIX_GUTT  + key

    jenter = jente_tall.get(jente_key, 0)
    gutter = gutt_tall.get(gutt_key,  0)
    totalt = jenter + gutter

    if totalt == 0:
        return "Ukjent"

    andel_min = min(jenter, gutter) / totalt

    if andel_min >= NEUTRAL_THRESHOLD:
        return f"Kjønnsnøytralt (Kvinne: {jenter:,}, Mann: {gutter:,})"
    elif jenter >= gutter:
        return "Kvinne"
    else:
        return "Mann"


# Verifiser med kjente norske navn
test_navn = ["Emma", "Ole", "Kim", "Ingrid", "Andreas", "Robin", "Nora", "Tobias", "Alex", "Frida"]
print("Verifisering med kjente navn:")
for n in test_navn:
    print(f"  {n:<20} → {classify_gender(n)}")

## 6. Les CSV og legg til `kjønn`-kolonne

In [ ]:
df = pd.read_excel(INPUT_CSV) # Changed to pd.read_excel
df.columns = df.columns.str.strip().str.lower()

if "name" not in df.columns: # Changed 'navn' to 'name'
    raise ValueError(
        f"Fant ikke kolonnen 'name' i {INPUT_CSV}. " # Changed 'navn' to 'name'
        f"Tilgjengelige kolonner: {list(df.columns)}"
    )

print(f"Lastet {len(df)} rader fra '{INPUT_CSV}'")
df["kjønn"] = df["name"].apply(lambda n: classify_gender(str(n).split(' ')[0])) # Extract only the first name
df

## 7. Lagre resultat

In [ ]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"✓ Resultater lagret til '{OUTPUT_CSV}'")

In [ ]:
OUTPUT_EXCEL = "/content/drive/MyDrive/5900 Masteroppgave/navn_output.xlsx"
df.to_excel(OUTPUT_EXCEL, index=False)
print(f"✓ Resultater lagret til '{OUTPUT_EXCEL}'")

## 8. Oppsummering

In [ ]:
def forenklet(k):
    return "Kjønnsnøytralt" if k.startswith("Kjønnsnøytralt") else k

print("=== Fordeling ===")
print(df["kjønn"].apply(forenklet).value_counts().to_string())
print(f"\nTotalt: {len(df)} navn")

---

## Feilsøking

In [ ]:
# Sjekk et spesifikt navn
søk = "Emma"  # <-- endre dette
key = søk.upper()
print(f"Jente-kode '{PREFIX_JENTE + key}': {jente_tall.get(PREFIX_JENTE + key, 'IKKE FUNNET')}")
print(f"Gutt-kode  '{PREFIX_GUTT  + key}': {gutt_tall.get( PREFIX_GUTT  + key, 'IKKE FUNNET')}")

In [ ]:
ukjent_count = df[df['kjønn'] == 'Ukjent'].shape[0]
print(f"Antall rader med 'Ukjent' som kjønn: {ukjent_count}")

In [ ]:
ukjente_navn_df = df[df['kjønn'] == 'Ukjent']
print(f"Antall navn klassifisert som 'Ukjent': {len(ukjente_navn_df)}")
display(ukjente_navn_df.head())

---

## Notater

| | |
|---|---|
| **Datakilde** | SSB tabell [10467](https://www.ssb.no/statbank/table/10467) – Fødte etter fornavn, 1880–2025 |
| **Kodingslogikk** | SSB prefiks `1` = jente, `2` = gutt (f.eks. `1EMMA`, `2OLE`) |
| **Lisens** | [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) |
| **`Ukjent`** | Navn ikke i SSBs base (< 4 forekomster eller ikke brukt i Norge) |
| **`NEUTRAL_THRESHOLD`** | Juster i celle 2. Standard `0.25` = kjønnsnøytralt om minst 25 % tilhører ett kjønn |
| **API-grenser** | Maks 800 000 dataceller og 30 kall/minutt per IP |